# Notebook 05 — RQ3: Cross-Asset Dependencies

Granger causality network, DCC rolling correlations, and Diebold-Mariano test comparing own-asset vs cross-asset XGBoost models. Tests H0: assets are informationally independent.

**QM640 Data Analytics Capstone · Walsh College · Anupam K Mitra · May 2026**

---
> **Standalone notebook** — all code is self-contained. Run cells top-to-bottom. Data is downloaded automatically on first run.

## 1. Setup & Data

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import xgboost as xgb, yfinance as yf, time
warnings.filterwarnings('ignore')
%matplotlib inline

TICKERS = ['^NSEI','INFY.NS','SBIN.NS','GC=F',
           'USDINR=X','SPY','CL=F','^NSEBANK']
START, END = '2010-01-01','2026-04-30'
RF, COST   = 0.065, 20/10_000

dfs = {}
for t in TICKERS:
    time.sleep(0.3)
    try:
        raw = yf.download(t, start=START, end=END,
                           auto_adjust=True, progress=False)
        s = (raw['Close'] if isinstance(raw.columns, pd.MultiIndex)
             else raw.iloc[:,0]).squeeze()
        if s.notna().mean() > 0.3: dfs[t] = s
    except: pass

prices  = pd.DataFrame(dfs).ffill(limit=3)
log_ret = np.log(prices / prices.shift(1)).clip(-0.20, 0.20)
print(f'Tickers available: {list(prices.columns)}')

## 2. Granger Causality Network

In [ ]:
def granger_f(y, x, max_lag=5):
    mask = ~(np.isnan(y)|np.isnan(x))
    y, x = y[mask].astype(float), x[mask].astype(float)
    if len(y) < max_lag*4: return np.nan, 1.0

    def ols_rss(Y, X):
        Xc = np.hstack([np.ones((len(X),1)), X])
        b  = np.linalg.lstsq(Xc, Y, rcond=None)[0]
        return float(((Y - Xc@b)**2).sum())

    T  = len(y); p = max_lag
    Y_r = y[p:]; X_r = np.column_stack([y[p-k:T-k] for k in range(1,p+1)])
    Y_u = y[p:]; X_u = np.column_stack(
        [y[p-k:T-k] for k in range(1,p+1)] +
        [x[p-k:T-k] for k in range(1,p+1)])

    rss_r = ols_rss(Y_r, X_r); rss_u = ols_rss(Y_u, X_u)
    if rss_u <= 0 or rss_r <= rss_u: return np.nan, 1.0
    F = ((rss_r-rss_u)/p) / (rss_u/max(T-2*p-1,1))
    p_val = float(stats.f.sf(F, p, T-2*p-1))
    return round(F,4), round(p_val,6)

tickers = [c for c in log_ret.columns if log_ret[c].notna().sum() > 500]
rows = []
for cause in tickers:
    for target in tickers:
        if cause == target: continue
        F, p = granger_f(log_ret[target].values, log_ret[cause].values)
        rows.append({'cause':cause,'target':target,'F':F,'p_val':p})

gc_df = pd.DataFrame(rows)
# BH FDR correction
from statsmodels.stats.multitest import multipletests
reject, _, _, _ = multipletests(gc_df['p_val'].fillna(1), alpha=0.05, method='fdr_bh')
gc_df['reject_BH'] = reject
sig = gc_df[gc_df['reject_BH']].sort_values('F', ascending=False)

print(f'Granger pairs tested:        {len(gc_df)}')
print(f'BH-significant (q=0.05):     {gc_df["reject_BH"].sum()}')
print(f'H1 condition (≥3 pairs): {"CONFIRMED ✓" if gc_df["reject_BH"].sum() >= 3 else "fail"}')
print('\nTop 10 causal pairs:')
print(sig.head(10)[['cause','target','F','p_val']].to_string(index=False))

## 3. DCC Rolling Correlations

In [ ]:
key_pairs = [('USDINR=X','INFY.NS'),('SPY','^NSEI'),('GC=F','^NSEI')]
fig, axes = plt.subplots(len(key_pairs), 1, figsize=(13, 3.5*len(key_pairs)))
for ax, (a, b) in zip(axes, key_pairs):
    if a not in log_ret or b not in log_ret:
        ax.text(0.5,0.5,f'{a} or {b} not available',ha='center',transform=ax.transAxes)
        continue
    corr = log_ret[a].rolling(252).corr(log_ret[b])
    ax.fill_between(corr.index, corr.values, 0,
                    where=(corr.values>0), alpha=0.25, color='#1D9E75')
    ax.fill_between(corr.index, corr.values, 0,
                    where=(corr.values<0), alpha=0.25, color='#E24B4A')
    ax.plot(corr.index, corr, color='#1C3678', lw=1.2)
    ax.axhline(0, color='black', lw=0.6, ls='--')
    ax.set_title(f'{a} × {b} — 252-day Rolling Correlation', fontweight='bold')
    ax.set_ylim(-1,1); ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('../results/figures/05_dcc_correlations.png', dpi=150)
plt.show(); print('DCC plot saved')

## 4. Diebold-Mariano Test — Own vs Cross-Asset XGBoost

In [ ]:
TRAIN_W, TEST_W = 756, 63
TARGETS = [t for t in ['^NSEI','INFY.NS','SBIN.NS','GC=F']
           if t in log_ret.columns]

def walkforward_dm(X_df, y_ser):
    X_df = X_df.fillna(0); y_ser = y_ser.reindex(X_df.index)
    n = len(X_df); errors = []; actuals = []
    for s in range(TRAIN_W, n-TEST_W, TEST_W):
        Xtr=X_df.iloc[s-TRAIN_W:s]; ytr=y_ser.iloc[s-TRAIN_W:s]
        Xte=X_df.iloc[s:s+TEST_W];  yte=y_ser.iloc[s:s+TEST_W]
        mask = ytr.notna() & Xtr.notna().all(axis=1)
        if mask.sum() < 50: continue
        sc_ = StandardScaler()
        m = xgb.XGBRegressor(n_estimators=200,max_depth=3,learning_rate=0.05,
                              subsample=0.8,colsample_bytree=0.5,
                              random_state=42,verbosity=0)
        m.fit(sc_.fit_transform(Xtr[mask]), ytr[mask])
        pred = m.predict(sc_.transform(Xte.fillna(0)))
        errors.extend((yte.values - pred).tolist())
        actuals.extend(yte.values.tolist())
    return np.array(errors), np.array(actuals)

dm_results = []
for target in TARGETS:
    y = log_ret[target].shift(-5).rolling(5).sum() - COST*2

    # Own-asset features
    r = log_ret[target]
    X_own = pd.DataFrame({
        'lag1':r.shift(1),'lag2':r.shift(2),'lag3':r.shift(3),
        'vol21':r.rolling(21).std().shift(1),
        'mom63':np.log(prices[target]/prices[target].shift(63)).shift(1)
        if target in prices else r.rolling(63).sum().shift(1),
    }, index=log_ret.index)

    # Cross-asset features (add all other tickers)
    X_cross = X_own.copy()
    for other in log_ret.columns:
        if other == target: continue
        stub = other.replace('^','').replace('=','').replace('.','_')
        X_cross[f'{stub}_lag1'] = log_ret[other].shift(1)
        X_cross[f'{stub}_21d']  = log_ret[other].rolling(21).sum().shift(1)

    e1, a1 = walkforward_dm(X_own,   y)
    e2, a2 = walkforward_dm(X_cross, y)
    n = min(len(e1),len(e2))
    e1,e2,act = e1[-n:],e2[-n:],a2[-n:]
    d    = e1**2 - e2**2
    DM   = float(d.mean() / (d.std()/np.sqrt(n))) if d.std()>0 else 0
    p_val= float(stats.norm.sf(DM))
    rmse_red = (np.sqrt(np.mean(e1**2)) - np.sqrt(np.mean(e2**2))) / np.sqrt(np.mean(e1**2)) * 100
    dm_results.append({'target':target,'DM':round(DM,3),'p_val':round(p_val,4),
                        'RMSE_red%':round(rmse_red,2),'reject':p_val<0.05})
    print(f'{target:<18} DM={DM:>6.3f}  p={p_val:.4f}  '
          f'RMSE-red={rmse_red:+.2f}%  {"REJECT H0 ✓" if p_val<0.05 else "fail"}')

dm_df = pd.DataFrame(dm_results)
print(f'\nDM-significant: {dm_df["reject"].sum()}/{len(dm_df)}')
print(f'H1 (DM):        {"CONFIRMED ✓" if dm_df["reject"].any() else "NOT confirmed"}')